# modeling_v14 — V1 드리프트 피처 생성 (P1~P3 정적 · P4 규약)

**로드맵** V1 · **성격** 피처 생성(채점 없음) · **판정 축** 없음(생성 단계) · **다음** V2 시간축 스크리닝

> ⚠️ **실행 위치**: 이 노트북은 **`modeling_v14/` 폴더에서** 실행해줘(옆의 `v14_features.py` 임포트).
> ⚠️ **미러 미검증**: v14 방침대로 클라우드 실행검증 안 했어. 정적 검토 + 촘촘한 assert 로 보완했으니, 첫 실행 시 아래 '확인 포인트'를 봐줘.

## 하는 일
- 센서 집계(lean-85 기저 ~73개)를 **누수 0 상대값**으로 변환해 3종 생성:
  - **P1** PM 위상별 z-score(`__p1pmz`) · **P2** 레짐 baseline 차감(`__p2reg`) · **P3** 직전 k-Lot median 편차(`__p3dt`)
- 각 웨이퍼는 **자기보다 시각(wf_ts)이 엄격히 앞선 데이터만** 참조(expanding). 타깃 C65 는 절대 미사용.
- **P4**(타깃 분해)는 fold 안에서 core10 재학습이 필요 → 정적 컬럼화 불가. 규약 함수만 자기검증(실제 채점=V2).

## 전제 파일 (경로 자동탐색)
`v13_fdc_pool_wf_oof.csv.gz` · `core10_meta_wf.csv` · `lean_timestable_set.json` · `tuned_params_v13_xgb.json` · `train_data.csv` · **`v14_features.py`(같은 폴더)**

## 예상 런타임
로컬 venv 기준 **약 1~3분**(P1~P3 벡터화 + P4 한-split XGB 1회).

## 확인 포인트
- [ ] `[셀3]` P1/P2/P3 각 shape · **floor 5센서(C17·C11·C31·C15·C16) 각 ≥1** · NaN 0
- [ ] `[셀4]` **R11 인과 spot-check PASS**(P2 표본 재계산 일치=미래참조 없음)
- [ ] `[셀5]` `data/v14_P1.csv·v14_P2.csv·v14_P3.csv` 저장(각 11,939행)
- [ ] `[셀6]` P4 규약 자기검증 OK(예측 유한값)

## 산출물
`modeling_v14/data/v14_P1.csv`, `v14_P2.csv`, `v14_P3.csv` (C64 + 상대값 피처). → V2 가 시간축으로 채점.


In [1]:
# [셀1] import + 모듈 + 경로
import warnings; warnings.filterwarnings("ignore")
import os, sys, json, time
import numpy as np, pandas as pd

# v14_features 임포트 (노트북은 modeling_v14/ 에서 실행)
for _p in [os.getcwd(), os.path.dirname(os.getcwd())]:
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)
import v14_features as vf

FMT = "%Y-%m-%d %H:%M:%S.%f"
ID_COL, TARGET_COL, C20_COL = vf.ID_COL, vf.TARGET_COL, vf.C20_COL

def _find(name):
    for d in [".", "data", "colab_GA", "..", os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"), os.path.join("..", "modeling_v13", "colab_GA"),
              os.path.join("..", "문제1(하)"), "문제1(하)"]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz"); META = _find("core10_meta_wf.csv")
LEANJSON = _find("lean_timestable_set.json"); XGBJSON = _find("tuned_params_v13_xgb.json"); TRAIN = _find("train_data.csv")
print("v14_features 로드 OK | PM bins", vf.PM_BIN_EDGES, "| floor", vf.PROTECTED)

v14_features 로드 OK | PM bins [-inf, 1, 3, 7, 14, 30, inf] | floor ['C17', 'C11', 'C31', 'C15', 'C16']


In [2]:
# [셀2] 로드 + 웨이퍼 시각 병합 + 센서 기저 정의 (V0 승계)
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
tt = pd.read_csv(TRAIN, usecols=[ID_COL, "C40"]); tt[ID_COL] = tt[ID_COL].astype(str)
tt["_ts"] = pd.to_datetime(tt["C40"], format=FMT)
df = df.merge(tt.groupby(ID_COL)["_ts"].min().reset_index().rename(columns={"_ts": "wf_ts"}),
              on=ID_COL, how="inner").reset_index(drop=True)

LEAN = json.load(open(LEANJSON, encoding="utf-8"))["lean_features"]
SENSOR_BASE = vf.sensor_base_cols(LEAN)                     # lean-85 중 센서 집계(core10·비센서 제외)
have = vf.assert_floor(SENSOR_BASE, "SENSOR_BASE")          # 기저부터 floor 만족 확인
assert TARGET_COL not in SENSOR_BASE, "타깃 유입 금지"
print(f"[셀2] df {df.shape} | 센서기저 {len(SENSOR_BASE)}개 | floor {have} | "
      f"기간 {str(df['wf_ts'].min())[:10]}~{str(df['wf_ts'].max())[:10]}")

[셀2] df (11939, 668) | 센서기저 75개 | floor {'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1} | 기간 2018-12-01~2019-02-08


In [3]:
# [셀3] P1/P2/P3 생성 (누수 0 상대값) + floor 재확인
t0 = time.time()
P1 = vf.build_P1_pm_relative(df, SENSOR_BASE)               # PM 위상별 과거-only z
P2 = vf.build_P2_regime_conditional(df, SENSOR_BASE)        # 레짐 과거-only baseline 차감
P3 = vf.build_P3_stationary(df, SENSOR_BASE, k_lots=5)      # 직전 5 Lot median 편차
for nm, P in [("P1", P1), ("P2", P2), ("P3", P3)]:
    feats = [c for c in P.columns if c != ID_COL]
    h = vf.assert_floor(feats, nm)
    n_nan = int(P[feats].isna().sum().sum())
    assert n_nan == 0, f"{nm} NaN {n_nan} (fillna 누락?)"
    print(f"[셀3] {nm}: {P.shape} | floor {h} | NaN {n_nan} | 예시 {feats[0]}")
print(f"[셀3] 생성 완료 {time.time()-t0:.0f}s")

[셀3] P1: (11939, 76) | floor {'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1} | NaN 0 | 예시 C17_max_step4__p1pmz
[셀3] P2: (11939, 76) | floor {'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1} | NaN 0 | 예시 C17_max_step4__p2reg
[셀3] P3: (11939, 76) | floor {'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1} | NaN 0 | 예시 C17_max_step4__p3dt
[셀3] 생성 완료 4s


In [4]:
# [셀4] R11 인과 spot-check — P2 보호센서에서 표본 재계산 대조 (미래참조 0 확인)
probe = next(c for c in SENSOR_BASE if vf.sensor_of(c) == "C17")   # 온도(제일 중요)
n_chk = vf.assert_causal_P2(df, P2, probe, n=50)
print(f"[셀4] R11 인과 spot-check PASS: P2[{probe}] {n_chk}행 엄격-과거 재계산 일치 (미래참조 없음)")
print("      (P1·P3 도 동일한 _past_stats/직전-Lot 구성 = 엄격 과거만)")

[셀4] R11 인과 spot-check PASS: P2[C17_max_step4] 50행 엄격-과거 재계산 일치 (미래참조 없음)
      (P1·P3 도 동일한 _past_stats/직전-Lot 구성 = 엄격 과거만)


In [5]:
# [셀5] 저장 → V2 로 넘김
os.makedirs("data", exist_ok=True)
paths = {}
for nm, P in [("P1", P1), ("P2", P2), ("P3", P3)]:
    fp = os.path.join("data", f"v14_{nm}.csv")
    P.to_csv(fp, index=False); paths[nm] = (fp, P.shape)
for nm, (fp, sh) in paths.items():
    print(f"[셀5] 저장 {nm}: {fp} {sh}")

[셀5] 저장 P1: data\v14_P1.csv (11939, 76)
[셀5] 저장 P2: data\v14_P2.csv (11939, 76)
[셀5] 저장 P3: data\v14_P3.csv (11939, 76)


In [6]:
# [셀6] P4 규약 자기검증 — 한 split 에서 코드 동작+인과만 (RMSE 채점은 V2, R7)
import xgboost as xgb
xgj = json.load(open(XGBJSON, encoding="utf-8"))
def make_model():
    p = dict(xgj["params"]); p.update(objective="reg:squarederror", tree_method="hist",
             device="cpu", random_state=42, n_estimators=int(xgj["n_estimators"]))
    return xgb.XGBRegressor(**p)

CORE = [c for c in vf.CORE10 if c in df.columns]
y = df[TARGET_COL].to_numpy(float)
lot_ts = df.groupby(C20_COL)["wf_ts"].transform("median")
init_end = df["wf_ts"].min() + pd.Timedelta(weeks=4)
tr = np.where((lot_ts <= init_end).to_numpy())[0]
te = np.where((lot_ts > init_end).to_numpy())[0][:200]        # 한 조각만(sanity)
pred, level = vf.p4_two_stage_fold(tr, te, df[CORE], df[SENSOR_BASE], y, make_model)
assert len(pred) == len(te) and np.isfinite(pred).all(), "P4 예측 이상"
print(f"[셀6] P4 규약 자기검증 OK: train {len(tr)} → test {len(te)}, 예측 유한값 {pred.shape}. "
      f"(stage1=core10 train만, 잔차 채점=V2)")
print("\n=== V1 완료 — P1~P3 저장됨, P4 규약 준비. 다음=V2 시간축 스크리닝 ===")

[셀6] P4 규약 자기검증 OK: train 4661 → test 200, 예측 유한값 (200,). (stage1=core10 train만, 잔차 채점=V2)

=== V1 완료 — P1~P3 저장됨, P4 규약 준비. 다음=V2 시간축 스크리닝 ===
